In [5]:
import pandas as pd

# Load the dataset
file_path = '/content/ecommerce_customer_churn_dataset.csv'
df = pd.read_csv(file_path)

# Display the first few rows and basic info
display(df.head())
print(df.info())

FileNotFoundError: [Errno 2] No such file or directory: '/content/ecommerce_customer_churn_dataset.csv'

In [ ]:
# Calculate percentage of missing values per column
missing_percent = (df.isnull().sum() / len(df)) * 100

# Create a summary dataframe
missing_summary = pd.DataFrame({'Missing Values': df.isnull().sum(), 'Percentage (%)': missing_percent})

# Filter to show only columns with missing values and sort descending
display(missing_summary[missing_summary['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False))

In [ ]:
import scipy.stats as stats

# 1. Calculate Skewness and Outliers using IQR
num_df = df.select_dtypes(include=[np.number])
skewness = num_df.skew().sort_values(ascending=False)

outlier_report = {}
for col in num_df.columns:
    Q1 = num_df[col].quantile(0.25)
    Q3 = num_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = num_df[(num_df[col] < lower_bound) | (num_df[col] > upper_bound)]
    outlier_report[col] = len(outliers)

# 2. Display results
analysis_df = pd.DataFrame({
    'Skewness': skewness,
    'Outlier Count (IQR)': pd.Series(outlier_report)
}).sort_values(by='Skewness', ascending=False)

print("Professional Skewness & Outlier Analysis:")
display(analysis_df)

In [ ]:
# 3. Visualize Outliers and Skewness for top 4 affected features
top_cols = analysis_df.head(4).index.tolist()

fig, axes = plt.subplots(len(top_cols), 2, figsize=(15, 5 * len(top_cols)))

for i, col in enumerate(top_cols):
    # Boxplot for Outliers
    sns.boxplot(x=df[col], ax=axes[i, 0], color='salmon')
    axes[i, 0].set_title(f'Boxplot: Outliers in {col}')

    # KDE for Skewness
    sns.histplot(df[col], kde=True, ax=axes[i, 1], color='teal')
    axes[i, 1].set_title(f'Distribution: Skewness of {col} ({df[col].skew():.2f})')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Correlation Heatmap
plt.figure(figsize=(16, 10))
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.show()

# 2. Target Variable Balance
plt.figure(figsize=(6, 4))
sns.countplot(x='Churned', data=df, palette='viridis')
plt.title('Distribution of Churn (Target Variable)')
plt.show()

# 3. Print Class Balance Percentage
print("Target Class Balance (%):")
print(df['Churned'].value_counts(normalize=True) * 100)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Advanced Density Analysis using Hexbin to solve overplotting
plt.figure(figsize=(12, 7))
hb = plt.hexbin(df['Age'], df['Average_Order_Value'], gridsize=30, cmap='YlGnBu', bins='log', mincnt=1)
cb = plt.colorbar(hb, label='Log10(Count)')

plt.title('Advanced Density Analysis: Average Order Value by Age', fontsize=14)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Average Order Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

### Enhanced Professional Conclusion: Data Integrity & Feature Independence

1.  **Spending Outliers are Age-Agnostic**: The high-value clusters (Top of the Y-axis) appear consistently across the 20-80 age range. This suggests that 'High Spenders' are driven by behavior or income levels, not by the customer's age.
2.  **Impossible Age Outliers**: The plot clearly shows data points at **Age 150 and 200**. These are biologically impossible and represent data entry errors or system glitches. These must be filtered out or capped to prevent model bias.
3.  **Low Correlation**: Because the density (dark blue) stays flat along the bottom for all ages, we've visually confirmed a near-zero linear correlation between Age and Order Value.
4.  **Action Plan**:
    *   Filter `Age <= 100` to fix the integrity issue.
    *   Treat `Average_Order_Value` with a Log Transform to bring 'Whale' customers into a manageable range for the model.

In [ ]:
# Investigate behavior and spending of impossible Age outliers (> 100)
age_outliers = df[df['Age'] > 100]

# Adding Average_Order_Value to the behavioral analysis
behavior_cols = ['Age', 'Average_Order_Value', 'Cart_Abandonment_Rate', 'Login_Frequency', 'Session_Duration_Avg', 'Customer_Service_Calls']

print(f"Found {len(age_outliers)} users with Age > 100. Displaying behavior and spending:")
display(age_outliers[behavior_cols].sort_values(by='Average_Order_Value', ascending=False))


- **Pattern Check**: If these 'old' users have normal login frequencies and service calls, they are likely real customers with a typo in the `Age` field.
- **Junk Check**: If these values are all zero or null, these records are likely system errors (synthetic noise) and should be dropped entirely rather than just clipped.

Now we should apply log Transformation on age column due to valid and normal pattern.


### **Mobile Engagement Analysis**
Investigating the relationship between `Mobile_App_Usage` (Binary) and behavioral engagement metrics.

In [ ]:
def plot_full_engagement_linear_analysis(original_df):
    # Create a copy for EDA
    eda_df = original_df.copy()

    # List of all three metrics to compare
    metrics = ['Session_Duration_Avg', 'Pages_Per_Session', 'Login_Frequency']

    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    for i, col in enumerate(metrics):
        # Plotting regression without manual dropna as requested
        # Note: sns.regplot handles NaNs internally by omitting them from the calculation
        sns.regplot(data=eda_df,
                    x='Mobile_App_Usage',
                    y=col,
                    ax=axes[i],
                    scatter_kws={'alpha':0.15, 'color':'teal'},
                    line_kws={'color':'red', 'linewidth': 2})

        # Calculate correlation for the title
        corr_val = eda_df['Mobile_App_Usage'].corr(eda_df[col])

        axes[i].set_title(f'{col.replace("_", " ")}\nCorrelation: {corr_val:.2f}', fontsize=14)
        axes[i].set_xlabel('Mobile App Usage (Frequency)', fontsize=12)
        axes[i].set_ylabel(col.replace("_", " "), fontsize=12)
        axes[i].grid(True, linestyle='--', alpha=0.4)

    plt.suptitle('Linear Impact of Mobile App Usage on Engagement Metrics', fontsize=18, y=1.05)
    plt.tight_layout()
    plt.show()

# Execute the full comparison
plot_full_engagement_linear_analysis(df)

With **correlations** of **0.76, 0.72, and 0.67** respectively, we can clearly see a **strong**, positive linear trend: as customers use the mobile app more frequently, their session duration, pages viewed, and login frequency all tend to rise significantly.